<a href="https://colab.research.google.com/github/esigelecParfait/Projet-Spark/blob/main/Partie3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
spark=SparkSession.builder.appName('practice').getOrCreate()
prestataire=spark.read.csv(r"/content/sample_data/Prestataire_1.csv",header=True,inferSchema=True)
prestataire.show(5)
villes=spark.read.csv(r"/content/sample_data/ville_1.csv",header=True,inferSchema=True)
villes.show(5)


+-------------+-------+-------+
|           id|largeur|hauteur|
+-------------+-------+-------+
|Prestataire_1|     50|     50|
+-------------+-------+-------+

+----+-------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+
|  id|     vitesse_a_pied|     vitesse_a_velo|                home|             travail|sportif|casseur|              statut|           salaire|sexe|age|         sportivite|velo_perf_minimale|
+----+-------------------+-------------------+--------------------+--------------------+-------+-------+--------------------+------------------+----+---+-------------------+------------------+
|5251|               0.02|               0.05|(lon:26.60 lat:28...|(lon:21.08 lat:14...|  false|  false|          reserviste|29800.610034665042|   F| 18|                0.1|               0.4|
|5252|0.14974625830876215|0.37436564577190534|(lon:0.26 lat:42.61)|

Non, l'affichage avec "show" n'est pas adapté.

In [ ]:
echantillon_villes=villes.sample(False,0.7,42)


In [ ]:
villes_pd=villes.toPandas()
villes_pd.head()

,id,vitesse_a_pied,vitesse_a_velo,home,travail,sportif,casseur,statut,salaire,sexe,age,sportivite,velo_perf_minimale
0,5251,0.020000,0.050000,(lon:26.60 lat:28.13),(lon:21.08 lat:14.11),False,False,reserviste,29800.610035,F,18,0.100000,0.4
1,5252,0.149746,0.374366,(lon:0.26 lat:42.61),(lon:36.35 lat:33.28),False,False,professeur,23595.443840,F,28,0.748731,0.4
2,5253,0.630971,1.682590,(lon:3.34 lat:13.95),(lon:24.75 lat:48.15),False,False,technicien_de_surface,18530.147763,H,65,2.103237,0.4
3,5254,0.040096,0.106923,(lon:19.54 lat:43.69),(lon:38.57 lat:42.65),False,False,technicien_de_surface,18997.602810,H,26,0.133653,0.4
4,5255,0.020000,0.050000,(lon:28.51 lat:41.70),(lon:17.67 lat:25.16),False,False,éboueur,23618.479750,F,50,0.100000,0.4


In [9]:
velos=spark.read.csv(r"/content/sample_data/velos/*.csv",header=True,inferSchema=True)
velos.show(5)
velos.count()

+--------+-------------------+-------------------+
| velo_id|        performance|          timestamp|
+--------+-------------------+-------------------+
|velo_221| 0.8475528879954313|2018-01-05 16:14:00|
|velo_221|0.14821009714870595|2018-01-06 20:23:00|
|velo_221|0.14658190917531475|2018-01-07 06:04:00|
|velo_221|0.14537679499231898|2018-01-07 08:51:00|
|velo_221|0.14305306724092204|2018-01-07 11:42:00|
+--------+-------------------+-------------------+
only showing top 5 rows


17529

In [10]:
stations=spark.read.csv(r"/content/sample_data/stations/*.csv",header=True,inferSchema=True)
stations.show(5)
stations.count()

+-------------------+----------+-----------+----------+-----------------+---------+
|          timestamp|station_id|cycliste_id|   velo_id| velo_performance|   action|
+-------------------+----------+-----------+----------+-----------------+---------+
|          timestamp|station_id|cycliste_id|   velo_id|             NULL|   action|
|          timestamp|station_id|performance|      NULL|             NULL|     NULL|
|2018-01-01 04:02:00|        24|       3188|velo_11132|0.936689745950615|recuperer|
|2018-01-01 04:02:00|        24|          1|      NULL|             NULL|     NULL|
|2018-01-01 08:28:00|        24|         77|velo_30447|             0.85|recuperer|
+-------------------+----------+-----------+----------+-----------------+---------+
only showing top 5 rows


2472

In [22]:
stations_17=spark.read.csv(r"/content/sample_data/stations/log_stations_17.csv",header=True,inferSchema=True)
stations_17.show(5)
stations_17.count()

+-------------------+----------+-----------+--------+----------------+------+
|          timestamp|station_id|cycliste_id| velo_id|velo_performance|action|
+-------------------+----------+-----------+--------+----------------+------+
|          timestamp|station_id|cycliste_id| velo_id|            NULL|action|
|          timestamp|station_id|performance|    NULL|            NULL|  NULL|
|2018-01-01 02:06:00|        17|       1531|velo_209|            0.85|donner|
|2018-01-01 02:06:00|        17|          1|    NULL|            NULL|  NULL|
|2018-01-01 10:58:00|        17|          1|    NULL|            NULL|  NULL|
+-------------------+----------+-----------+--------+----------------+------+
only showing top 5 rows


78

In [23]:
from pyspark.sql.functions import col
lignes_ecartes_17=stations_17.filter(col("station_id").isNull())
stations_prop=stations_17.exceptAll(lignes_ecartes_17)
lignes_ecart_17=stations_17.filter(col("velo_id").isNull())
stations_nettoyes_17=stations_prop.exceptAll(lignes_ecart_17)
stations_nettoyes_17.show(30)

+-------------------+----------+-----------+----------+-------------------+---------+
|          timestamp|station_id|cycliste_id|   velo_id|   velo_performance|   action|
+-------------------+----------+-----------+----------+-------------------+---------+
|2018-01-11 02:06:00|        17|       1531|  velo_219|               0.85|   donner|
|2018-01-11 12:50:00|        17|       4823| velo_2763| 0.5148958180742051|   donner|
|2018-01-10 02:06:00|        17|       1531| velo_2179| 0.4112589559465957|   donner|
|2018-01-12 18:39:00|        17|       6139| velo_2027|0.37442499789583433|   donner|
|2018-01-04 12:50:00|        17|       4823|  velo_210|               0.85|   donner|
|2018-01-03 02:06:00|        17|       1531|  velo_214|               0.85|   donner|
|2018-01-08 16:36:00|        17|       4339|  velo_213|               0.85|   donner|
|2018-01-13 05:59:00|        17|       3845| velo_2027| 0.3578112431146158|recuperer|
|2018-01-04 02:06:00|        17|       1531|  velo_218

In [25]:
lignes_entetes_17 = stations_nettoyes_17.filter(col("station_id") == "station_id")
stations_nettoyes_17 = stations_nettoyes_17.exceptAll(lignes_entetes_17)
stations_nettoyes_17.show(30, truncate=False)
stations_nettoyes_17.count()


+-------------------+----------+-----------+----------+-------------------+---------+
|timestamp          |station_id|cycliste_id|velo_id   |velo_performance   |action   |
+-------------------+----------+-----------+----------+-------------------+---------+
|2018-01-11 02:06:00|17        |1531       |velo_219  |0.85               |donner   |
|2018-01-11 12:50:00|17        |4823       |velo_2763 |0.5148958180742051 |donner   |
|2018-01-10 02:06:00|17        |1531       |velo_2179 |0.4112589559465957 |donner   |
|2018-01-12 18:39:00|17        |6139       |velo_2027 |0.37442499789583433|donner   |
|2018-01-04 12:50:00|17        |4823       |velo_210  |0.85               |donner   |
|2018-01-03 02:06:00|17        |1531       |velo_214  |0.85               |donner   |
|2018-01-08 16:36:00|17        |4339       |velo_213  |0.85               |donner   |
|2018-01-13 05:59:00|17        |3845       |velo_2027 |0.3578112431146158 |recuperer|
|2018-01-04 02:06:00|17        |1531       |velo_218  

35

Oui je constate des problèmes, les fichiers ont mal chargé dans mon dataframe, les données ne sont pas alignées de la bonne manière

Les fichiers csv de stations sont mal organisés ie l'entete est mal respecte dans ces fichiers, on retrouve plusieurs fois l'entete dans le fichier, et certains lignes n'ont pas la meme longueur correspondant au nombre de colonnes du fichier

In [19]:
from pyspark.sql.functions import col
lignes_ecartes=stations.filter(col("station_id").isNull())
stations_prop=stations.exceptAll(lignes_ecartes)
lignes_ecart=stations.filter(col("velo_id").isNull())
stations_nettoyes=stations_prop.exceptAll(lignes_ecart)
stations_nettoyes.show(30)

+-------------------+----------+-----------+----------+-------------------+---------+
|          timestamp|station_id|cycliste_id|   velo_id|   velo_performance|   action|
+-------------------+----------+-----------+----------+-------------------+---------+
|2018-01-09 10:57:00|        24|        381| velo_1296| 0.4894505987898065|recuperer|
|2018-01-07 10:02:00|        13|        760|velo_23728|-0.5101449050917827|recuperer|
|2018-01-01 18:55:00|        23|         42|  velo_345|               0.85|   donner|
|2018-01-10 17:02:00|        18|       4330|  velo_899| 0.6604631097918819|recuperer|
|2018-01-14 09:35:00|         2|       5810|velo_13985| 0.5067667756135378|   donner|
|2018-01-07 10:41:00|        19|        568| velo_1393| 0.7374818247520174|recuperer|
|2018-01-13 10:43:00|        14|       2565|  velo_165|               0.85|   donner|
|2018-01-10 09:49:00|        24|        381|velo_11683| 0.2727826387223332|   donner|
|2018-01-11 17:29:00|        24|         77| velo_2984

In [21]:
lignes_entetes = stations_nettoyes.filter(col("station_id") == "station_id")
stations_nettoyes = stations_nettoyes.exceptAll(lignes_entetes)
stations_nettoyes.show(30, truncate=False)


+-------------------+----------+-----------+----------+-------------------+---------+
|timestamp          |station_id|cycliste_id|velo_id   |velo_performance   |action   |
+-------------------+----------+-----------+----------+-------------------+---------+
|2018-01-09 10:57:00|24        |381        |velo_1296 |0.4894505987898065 |recuperer|
|2018-01-07 10:02:00|13        |760        |velo_23728|-0.5101449050917827|recuperer|
|2018-01-01 18:55:00|23        |42         |velo_345  |0.85               |donner   |
|2018-01-10 17:02:00|18        |4330       |velo_899  |0.6604631097918819 |recuperer|
|2018-01-14 09:35:00|2         |5810       |velo_13985|0.5067667756135378 |donner   |
|2018-01-07 10:41:00|19        |568        |velo_1393 |0.7374818247520174 |recuperer|
|2018-01-13 10:43:00|14        |2565       |velo_165  |0.85               |donner   |
|2018-01-10 09:49:00|24        |381        |velo_11683|0.2727826387223332 |donner   |
|2018-01-11 17:29:00|24        |77         |velo_2984 